# Edge-Native Hybrid ChakraModel Evaluation
This notebook evaluates the fully combined YOLO + ChakraTransformer architecture across all datasets using cloud GPUs.

In [ ]:
!pip install -q ultralytics timm opencv-python-headless torchvision gudhi

import sys
import os
import glob
import shutil
from pathlib import Path

print("Looking for attached code dataset...")
# Deep search for evaluate_all.py across all nested Kaggle folders
eval_script = glob.glob('/kaggle/input/**/src/evaluate_all.py', recursive=True)

if eval_script:
    # The parent of the 'src' folder is our root codebase directory
    src_dir = os.path.dirname(os.path.dirname(eval_script[0]))
    print(f"Found codebase at: {src_dir}")
    print("Copying to working directory to allow execution...")
    
    if os.path.exists('/kaggle/working/src'): shutil.rmtree('/kaggle/working/src')
    if os.path.exists('/kaggle/working/weights'): shutil.rmtree('/kaggle/working/weights')
    
    # Copy source and weights
    shutil.copytree(os.path.join(src_dir, 'src'), '/kaggle/working/src')
    shutil.copytree(os.path.join(src_dir, 'weights'), '/kaggle/working/weights')
    sys.path.insert(0, '/kaggle/working/src')
    print("✅ Setup Complete!")
else:
    print("ERROR: Could not find 'src/evaluate_all.py'.")
    print("Check the right sidebar -> 'Data' -> 'Datasets' to ensure you attached the Code Dataset!")


## Setup Datasets


In [ ]:
os.makedirs('/kaggle/working/data', exist_ok=True)

def auto_symlink(dataset_name, internal_name):
    # Find any folder in /kaggle/input that contains dataset_name (case-insensitive)
    all_dirs = glob.glob('/kaggle/input/**/', recursive=True)
    
    ds_dirs = [d for d in all_dirs if dataset_name.lower() in d.lower()]
    
    target_dir = None
    for d in ds_dirs:
        subdirs = [os.path.basename(os.path.normpath(sub)).lower() for sub in glob.glob(d + '/*/')]
        if 'images' in subdirs or 'original' in subdirs or 'image' in subdirs:
            target_dir = d
            break
            
    if target_dir:
        target = f'/kaggle/working/data/{internal_name}'
        if not os.path.exists(target):
            os.symlink(target_dir, target)
            print(f'✅ Linked {internal_name} from {target_dir}')
    else:
        print(f'⚠️ WARNING: {internal_name} not found.')
        found = [d for d in all_dirs if dataset_name.lower() in d.lower()]
        if found: print(f"   Found related paths but no 'images' or 'original' folder inside: {found}")

auto_symlink('kvasir', 'kvasir-seg')
auto_symlink('clinic', 'cvc-clinicdb')
auto_symlink('etis', 'etis-larib')
auto_symlink('colon', 'cvc-colondb')
auto_symlink('300', 'cvc-300')


## Run Final Evaluation
This will run `evaluate_all.py` which loads both the YOLO detection model and the ChakraTransformer segmentation model, simulating the Edge-Native Hybrid architecture pipeline!

In [ ]:
!python /kaggle/working/src/evaluate_all.py